In [ ]:
# Install Kaggle API
!pip install kaggle

# Import necessary libraries
import os
import zipfile
from google.colab import files

# Upload your kaggle.json file
print("Please upload your kaggle.json file")
uploaded = files.upload()

# Move kaggle.json to the correct location
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the Kaggle dataset
!kaggle datasets download -d techsash/waste-classification-data

# Extract the Kaggle dataset
with zipfile.ZipFile('waste-classification-data.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset')
print("Kaggle dataset extracted to 'dataset/DATASET/'")

# Verify the dataset directory
data_dir = 'dataset/DATASET/'
if not os.path.exists(data_dir):
    raise FileNotFoundError("Kaggle dataset extraction failed. The 'dataset/DATASET/' directory does not exist.")

# Upload the scraped data ZIP files
print("Please upload 'recyclable_images.zip' and 'non_recyclable_images.zip'")
uploaded = files.upload()

# Verify the scraped data ZIP files
recyclable_zip = '/content/recyclable_images.zip'
non_recyclable_zip = '/content/non_recyclable_images.zip'
if not os.path.exists(recyclable_zip) or not os.path.exists(non_recyclable_zip):
    raise FileNotFoundError("Please ensure both 'recyclable_images.zip' and 'non_recyclable_images.zip' were uploaded correctly.")

Please upload your kaggle.json file


Dataset URL: https://www.kaggle.com/datasets/techsash/waste-classification-data
License(s): CC-BY-SA-4.0
Kaggle dataset extracted to 'dataset/DATASET/'
Please upload 'recyclable_images.zip' and 'non_recyclable_images.zip'


In [ ]:
# Create directories for external data
os.makedirs('dataset/DATASET/TRAIN/R/external', exist_ok=True)
os.makedirs('dataset/DATASET/TRAIN/O/external', exist_ok=True)

# Import the shutil module
import shutil

# Extract the scraped data
with zipfile.ZipFile(recyclable_zip, 'r') as zip_ref:
    zip_ref.extractall('dataset/DATASET/TRAIN/R/external/')
with zipfile.ZipFile(non_recyclable_zip, 'r') as zip_ref:
    zip_ref.extractall('dataset/DATASET/TRAIN/O/external/')

# Flatten subfolders with duplicate handling
for root, dirs, files_list in os.walk('dataset/DATASET/TRAIN/R/external'):
    for f in files_list:
        if f.endswith(('.jpg', '.jpeg', '.png')):
            target_path = f'dataset/DATASET/TRAIN/R/external/{f}'
            base_name, ext = os.path.splitext(f)
            counter = 1
            while os.path.exists(target_path):
                new_filename = f"{base_name}_{counter}{ext}"
                target_path = f'dataset/DATASET/TRAIN/R/external/{new_filename}'
                counter += 1
            shutil.move(os.path.join(root, f), target_path)

for root, dirs, files_list in os.walk('dataset/DATASET/TRAIN/O/external'):
    for f in files_list:
        if f.endswith(('.jpg', '.jpeg', '.png')):
            target_path = f'dataset/DATASET/TRAIN/O/external/{f}'
            base_name, ext = os.path.splitext(f)
            counter = 1
            while os.path.exists(target_path):
                new_filename = f"{base_name}_{counter}{ext}"
                target_path = f'dataset/DATASET/TRAIN/O/external/{new_filename}'
                counter += 1
            shutil.move(os.path.join(root, f), target_path)

# Count images in the external directories
recyclable_count = len([f for f in os.listdir('dataset/DATASET/TRAIN/R/external') if f.endswith(('.jpg', '.jpeg', '.png'))])
non_recyclable_count = len([f for f in os.listdir('dataset/DATASET/TRAIN/O/external') if f.endswith(('.jpg', '.jpeg', '.png'))])
print(f"External directories: {recyclable_count} recyclable and {non_recyclable_count} non-recyclable images")

External directories: 500 recyclable and 500 non-recyclable images


In [ ]:
# Save the entire labeled dataset (dataset/DATASET/) as a ZIP file
zip_path = '/content/labeled_waste_dataset_no_taco.zip'
shutil.make_archive('/content/labeled_waste_dataset_no_taco', 'zip', 'dataset/DATASET')
print(f"Labeled dataset saved as {zip_path}")

# Download the ZIP file to your local system
from google.colab import files as colab_files
colab_files.download(zip_path)

Labeled dataset saved as /content/labeled_waste_dataset_no_taco.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Install efficientnet-pytorch
#!pip install efficientnet-pytorch

# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from efficientnet_pytorch import EfficientNet
import os

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define data transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Define the dataset directory
data_dir = 'dataset/DATASET/'

# Create datasets
image_datasets = {
    'train': datasets.ImageFolder(os.path.join(data_dir, 'TRAIN'), data_transforms['train']),
    'test': datasets.ImageFolder(os.path.join(data_dir, 'TEST'), data_transforms['test'])
}

# Create data loaders
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=32, shuffle=True, num_workers=2),
    'test': DataLoader(image_datasets['test'], batch_size=32, shuffle=False, num_workers=2)
}

# Get class names and dataset sizes
class_names = image_datasets['train'].classes  # Should be ['O', 'R']
print(f"Class names: {class_names}")
print(f"Training samples (ImageFolder): {len(image_datasets['train'])} (including scraped images)")
print(f"Test samples: {len(image_datasets['test'])}")

# Recursively count images in each class for training
def count_images(directory):
    count = 0
    for root, dirs, files in os.walk(directory):
        for f in files:
            if f.endswith(('.jpg', '.jpeg', '.png')):
                count += 1
    return count

train_recyclable = count_images(os.path.join(data_dir, 'TRAIN/R'))
train_non_recyclable = count_images(os.path.join(data_dir, 'TRAIN/O'))
print(f"Training recyclable (R): {train_recyclable} images")
print(f"Training non-recyclable (O): {train_non_recyclable} images")

# Verify total matches ImageFolder
total_calculated = train_recyclable + train_non_recyclable
print(f"Total calculated training samples: {total_calculated}")
if total_calculated != len(image_datasets['train']):
    print(f"Warning: Calculated total ({total_calculated}) does not match ImageFolder total ({len(image_datasets['train'])}). There may be non-image files or hidden files.")

# Compute class weights to handle imbalance
total_samples = train_recyclable + train_non_recyclable
class_weights = torch.tensor([
    total_samples / (2 * train_non_recyclable),  # Weight for class O (non-recyclable)
    total_samples / (2 * train_recyclable)       # Weight for class R (recyclable)
]).to(device)
print(f"Class weights: {class_weights}")

# Load pretrained EfficientNet-B0
model = EfficientNet.from_pretrained('efficientnet-b0')
print("Loaded pretrained weights for efficientnet-b0")

# Modify the classifier head for binary classification (2 classes: recyclable vs. non-recyclable)
num_ftrs = model._fc.in_features
model._fc = nn.Linear(num_ftrs, 2)  # 2 classes: O (non-recyclable), R (recyclable)

# Move the model to the GPU
model = model.to(device)

# Define loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Define a learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# Training function
def train_model(model, dataloaders, criterion, optimizer, scheduler, num_epochs=10):
    best_acc = 0.0
    best_model_wts = model.state_dict()

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'test']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Deep copy the model if it has the best accuracy
            if phase == 'test' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()

        print()

    print(f'Best test Acc: {best_acc:.4f}')

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model

# Train the model
num_epochs = 10
model = train_model(model, dataloaders, criterion, optimizer, scheduler, num_epochs=num_epochs)

# Save the trained model as a .pth file
torch.save(model.state_dict(), '/content/waste_classification_model.pth')
print("Model saved as /content/waste_classification_model.pth")

# Download the model to your local system
from google.colab import files as colab_files
colab_files.download('/content/waste_classification_model.pth')

Using device: cuda
Class names: ['O', 'R']
Training samples (ImageFolder): 23564 (including scraped images)
Test samples: 2513
Training recyclable (R): 10499 images
Training non-recyclable (O): 13065 images
Total calculated training samples: 23564
Class weights: tensor([0.9018, 1.1222], device='cuda:0')
Loaded pretrained weights for efficientnet-b0
Loaded pretrained weights for efficientnet-b0
Epoch 1/10
----------


/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


train Loss: 0.3127 Acc: 0.8711
test Loss: 0.3151 Acc: 0.8854

Epoch 2/10
----------


/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


train Loss: 0.2809 Acc: 0.8847
test Loss: 0.1976 Acc: 0.9320

Epoch 3/10
----------


...

train Loss: 0.1834 Acc: 0.9251
test Loss: 0.2118 Acc: 0.9276

Epoch 10/10
----------


/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


train Loss: 0.1779 Acc: 0.9271
test Loss: 0.2496 Acc: 0.9184

Best test Acc: 0.9415
Model saved as /content/waste_classification_model.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import necessary libraries
import os
import shutil

# Create a directory in Google Drive to store the project files
project_dir = '/content/drive/MyDrive/WasteClassificationProject'
os.makedirs(project_dir, exist_ok=True)
print(f"Project directory created at: {project_dir}")

# Define the source dataset directory (assumes dataset/DATASET/ exists in Colab)
data_dir = 'dataset/DATASET/'
if not os.path.exists(data_dir):
    raise FileNotFoundError("The 'dataset/DATASET/' directory does not exist. Please ensure the dataset is prepared.")

# Copy the combined dataset to Google Drive
drive_data_dir = os.path.join(project_dir, 'dataset/DATASET')
shutil.copytree(data_dir, drive_data_dir)
print(f"Combined dataset saved to Google Drive at: {drive_data_dir}")

Mounted at /content/drive
Project directory created at: /content/drive/MyDrive/WasteClassificationProject
Combined dataset saved to Google Drive at: /content/drive/MyDrive/WasteClassificationProject/dataset/DATASET


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define the project directory in Google Drive
project_dir = '/content/drive/MyDrive/WasteClassificationProject'
os.makedirs(project_dir, exist_ok=True)

# Copy the .pth file to Google Drive
pth_path = '/content/waste_classification_model.pth'
drive_pth_path = os.path.join(project_dir, 'waste_classification_model.pth')
shutil.copy(pth_path, drive_pth_path)
print(f"Model .pth file saved to Google Drive at: {drive_pth_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model .pth file saved to Google Drive at: /content/drive/MyDrive/WasteClassificationProject/waste_classification_model.pth


In [ ]:
!pip install efficientnet_pytorch # Install the correct module 'efficientnet_pytorch'

  Preparing metadata (setup.py) ... done
  Created wheel for efficientnet_pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16424 sha256=65304aa3eb5c4f714cd64f42aed520f031da5de2c32a50e2d3b9e2c732b950f1
  Stored in directory: /root/.cache/pip/wheels/8b/6f/9b/231a832f811ab6ebb1b32455b177ffc6b8b1cd8de19de70c09
Successfully built efficientnet_pytorch


In [ ]:
import torch
from efficientnet_pytorch import EfficientNet

# Load the EfficientNet-B0 model
model = EfficientNet.from_pretrained('efficientnet-b0')
num_classes = 2  # Change based on your dataset (Recyclable, Non-Recyclable)
model._fc = torch.nn.Linear(model._fc.in_features, num_classes)  # Adjust the final layer

# Load trained model weights
model_path = "/content/waste_classification_model.pth"
model.load_state_dict(torch.load(model_path, map_location=torch.device("cpu")))

# Set model to evaluation mode
model.eval()
print("✅ EfficientNet Model Loaded Successfully!")


Loaded pretrained weights for efficientnet-b0
✅ EfficientNet Model Loaded Successfully!


In [ ]:
!pip install gradio torch torchvision numpy pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.7 MB/s eta 0:00:00


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import necessary libraries
import os

# Define the dataset directory in Google Drive
data_dir = '/content/drive/MyDrive/WasteClassificationProject/dataset/DATASET'

# Verify the dataset directory exists
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"Dataset directory not found at: {data_dir}. Please ensure the dataset is saved in Google Drive.")

# Verify the structure of the dataset
train_dir = os.path.join(data_dir, 'TRAIN')
test_dir = os.path.join(data_dir, 'TEST')
if not os.path.exists(train_dir) or not os.path.exists(test_dir):
    raise FileNotFoundError("TRAIN and TEST directories not found in the dataset.")

# Count images in each class for training
def count_images(directory):
    count = 0
    for root, dirs, files in os.walk(directory):
        for f in files:
            if f.endswith(('.jpg', '.jpeg', '.png')):
                count += 1
    return count

train_recyclable = count_images(os.path.join(train_dir, 'R'))
train_non_recyclable = count_images(os.path.join(train_dir, 'O'))
print(f"Training recyclable (R): {train_recyclable} images")
print(f"Training non-recyclable (O): {train_non_recyclable} images")
print(f"Total training images: {train_recyclable + train_non_recyclable}")

Mounted at /content/drive
Training recyclable (R): 10499 images
Training non-recyclable (O): 12568 images
Total training images: 23067


In [ ]:
# Import necessary libraries
import os

# Define the dataset directory
data_dir = '/content/drive/MyDrive/WasteClassificationProject/dataset/DATASET'

# Check if the TEST directory exists
test_dir = os.path.join(data_dir, 'TEST')
if not os.path.exists(test_dir):
    print(f"TEST directory not found at: {test_dir}")
else:
    print(f"TEST directory found at: {test_dir}")

# List the contents of the TEST directory
if os.path.exists(test_dir):
    test_contents = os.listdir(test_dir)
    print(f"Contents of TEST directory: {test_contents}")

    # Check for class subfolders (O and R)
    expected_classes = ['O', 'R']
    test_subfolders = [d for d in test_contents if os.path.isdir(os.path.join(test_dir, d))]
    print(f"Subfolders in TEST directory: {test_subfolders}")

    # Verify that O and R subfolders exist
    for cls in expected_classes:
        cls_path = os.path.join(test_dir, cls)
        if not os.path.exists(cls_path):
            print(f"Class folder {cls} not found in TEST directory.")
        else:
            # Count images in each class
            num_images = len([f for f in os.listdir(cls_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
            print(f"Class {cls} has {num_images} images in TEST directory.")

TEST directory found at: /content/drive/MyDrive/WasteClassificationProject/dataset/DATASET/TEST
Contents of TEST directory: []
Subfolders in TEST directory: []
Class folder O not found in TEST directory.
Class folder R not found in TEST directory.


In [ ]:
# Install the Kaggle API (if not already installed)
!pip install kaggle

# Import necessary libraries
import os
import zipfile
import shutil

# Configure Kaggle API (you need to upload kaggle.json)
print("Please upload your kaggle.json file:")
from google.colab import files
uploaded = files.upload()

# Move kaggle.json to the correct location
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# Download the Kaggle dataset
!kaggle datasets download -d techsash/waste-classification-data

# Extract the dataset to a temporary location
with zipfile.ZipFile('waste-classification-data.zip', 'r') as zip_ref:
    zip_ref.extractall('temp_dataset')

# Verify the extracted structure
extracted_dir = 'temp_dataset/DATASET'
if not os.path.exists(extracted_dir):
    raise FileNotFoundError("DATASET directory not found in the extracted files.")

# Check for TRAIN and TEST directories
train_dir = os.path.join(extracted_dir, 'TRAIN')
test_dir = os.path.join(extracted_dir, 'TEST')
print(f"TRAIN directory exists: {os.path.exists(train_dir)}")
print(f"TEST directory exists: {os.path.exists(test_dir)}")

# Check for class subfolders in TEST
if os.path.exists(test_dir):
    test_subfolders = [d for d in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, d))]
    print(f"Subfolders in TEST directory: {test_subfolders}")

    # Count images in each class
    for cls in ['O', 'R']:
        cls_path = os.path.join(test_dir, cls)
        if os.path.exists(cls_path):
            num_images = len([f for f in os.listdir(cls_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
            print(f"Class {cls} has {num_images} images in TEST directory.")
        else:
            print(f"Class {cls} not found in TEST directory.")

# Copy the TEST directory to Google Drive
drive_data_dir = '/content/drive/MyDrive/WasteClassificationProject/dataset/DATASET'
if os.path.exists(test_dir):
    drive_test_dir = os.path.join(drive_data_dir, 'TEST')
    if os.path.exists(drive_test_dir):
        shutil.rmtree(drive_test_dir)  # Remove the existing (empty) TEST directory
    shutil.copytree(test_dir, drive_test_dir)
    print(f"Copied TEST directory to Google Drive: {drive_test_dir}")
else:
    raise FileNotFoundError("TEST directory not found in the extracted dataset.")

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/techsash/waste-classification-data
License(s): CC-BY-SA-4.0
TRAIN directory exists: True
TEST directory exists: True
Subfolders in TEST directory: ['R', 'O']
Class O has 1401 images in TEST directory.
Class R has 1112 images in TEST directory.
Copied TEST directory to Google Drive: /content/drive/MyDrive/WasteClassificationProject/dataset/DATASET/TEST


In [ ]:
# Import necessary libraries
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define the dataset directory
data_dir = '/content/drive/MyDrive/WasteClassificationProject/dataset/DATASET'

# Define data transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),  # Resize without cropping
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(45),  # Increased rotation
        transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),  # Aggressive jitter
        transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), scale=(0.8, 1.2)),  # Translation and scaling
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),  # Resize without cropping
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Create datasets
image_datasets = {
    'train': datasets.ImageFolder(os.path.join(data_dir, 'TRAIN'), data_transforms['train']),
    'test': datasets.ImageFolder(os.path.join(data_dir, 'TEST'), data_transforms['test'])
}

# Create data loaders
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=32, shuffle=True, num_workers=2),
    'test': DataLoader(image_datasets['test'], batch_size=32, shuffle=False, num_workers=2)
}

# Get class names and dataset sizes
class_names = image_datasets['train'].classes  # Should be ['O', 'R']
print(f"Class names: {class_names}")
print(f"Training samples: {len(image_datasets['train'])}")
print(f"Test samples: {len(image_datasets['test'])}")

Using device: cuda
Class names: ['O', 'R']
Training samples: 23067
Test samples: 2513


In [ ]:
# Define the dataset directory
data_dir = '/content/drive/MyDrive/WasteClassificationProject/dataset/DATASET'

# Check the updated TEST directory
test_dir = os.path.join(data_dir, 'TEST')
if not os.path.exists(test_dir):
    print(f"TEST directory not found at: {test_dir}")
else:
    print(f"TEST directory found at: {test_dir}")

# List the contents of the TEST directory
if os.path.exists(test_dir):
    test_contents = os.listdir(test_dir)
    print(f"Contents of TEST directory: {test_contents}")

    # Check for class subfolders (O and R)
    expected_classes = ['O', 'R']
    test_subfolders = [d for d in test_contents if os.path.isdir(os.path.join(test_dir, d))]
    print(f"Subfolders in TEST directory: {test_subfolders}")

    # Verify that O and R subfolders exist
    for cls in expected_classes:
        cls_path = os.path.join(test_dir, cls)
        if not os.path.exists(cls_path):
            print(f"Class folder {cls} not found in TEST directory.")
        else:
            num_images = len([f for f in os.listdir(cls_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
            print(f"Class {cls} has {num_images} images in TEST directory.")

TEST directory found at: /content/drive/MyDrive/WasteClassificationProject/dataset/DATASET/TEST
Contents of TEST directory: ['R', 'O']
Subfolders in TEST directory: ['R', 'O']
Class O has 1401 images in TEST directory.
Class R has 1112 images in TEST directory.


In [ ]:
# Import necessary libraries
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define the dataset directory
data_dir = '/content/drive/MyDrive/WasteClassificationProject/dataset/DATASET'

# Define data transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),  # Resize without cropping
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(45),  # Increased rotation
        transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),  # Aggressive jitter
        transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), scale=(0.8, 1.2)),  # Translation and scaling
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),  # Resize without cropping
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Create datasets
image_datasets = {
    'train': datasets.ImageFolder(os.path.join(data_dir, 'TRAIN'), data_transforms['train']),
    'test': datasets.ImageFolder(os.path.join(data_dir, 'TEST'), data_transforms['test'])
}

# Create data loaders
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=32, shuffle=True, num_workers=2),
    'test': DataLoader(image_datasets['test'], batch_size=32, shuffle=False, num_workers=2)
}

# Get class names and dataset sizes
class_names = image_datasets['train'].classes  # Should be ['O', 'R']
print(f"Class names: {class_names}")
print(f"Training samples: {len(image_datasets['train'])}")
print(f"Test samples: {len(image_datasets['test'])}")

Using device: cuda
Class names: ['O', 'R']
Training samples: 23067
Test samples: 2513


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install required libraries
!pip install efficientnet-pytorch

# Import necessary libraries
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define the dataset directory
data_dir = '/content/drive/MyDrive/WasteClassificationProject/dataset/DATASET'

# Verify the dataset structure
train_dir = os.path.join(data_dir, 'TRAIN')
test_dir = os.path.join(data_dir, 'TEST')
if not os.path.exists(train_dir) or not os.path.exists(test_dir):
    raise FileNotFoundError("TRAIN and TEST directories not found in the dataset.")

# Count images in each class for training
def count_images(directory):
    count = 0
    for root, dirs, files in os.walk(directory):
        for f in files:
            if f.endswith(('.jpg', '.jpeg', '.png')):
                count += 1
    return count

train_recyclable = count_images(os.path.join(train_dir, 'R'))
train_non_recyclable = count_images(os.path.join(train_dir, 'O'))
print(f"Training recyclable (R): {train_recyclable} images")
print(f"Training non-recyclable (O): {train_non_recyclable} images")
print(f"Total training images: {train_recyclable + train_non_recyclable}")

# Count images in each class for testing
test_recyclable = count_images(os.path.join(test_dir, 'R'))
test_non_recyclable = count_images(os.path.join(test_dir, 'O'))
print(f"Test recyclable (R): {test_recyclable} images")
print(f"Test non-recyclable (O): {test_non_recyclable} images")
print(f"Total test images: {test_recyclable + test_non_recyclable}")

# Define data transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),  # Resize without cropping
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(45),  # Increased rotation
        transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),  # Aggressive jitter
        transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), scale=(0.8, 1.2)),  # Translation and scaling
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),  # Resize without cropping
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Create datasets
image_datasets = {
    'train': datasets.ImageFolder(os.path.join(data_dir, 'TRAIN'), data_transforms['train']),
    'test': datasets.ImageFolder(os.path.join(data_dir, 'TEST'), data_transforms['test'])
}

# Create data loaders
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=32, shuffle=True, num_workers=2),
    'test': DataLoader(image_datasets['test'], batch_size=32, shuffle=False, num_workers=2)
}

# Get class names and dataset sizes
class_names = image_datasets['train'].classes  # Should be ['O', 'R']
print(f"Class names: {class_names}")
print(f"Training samples: {len(image_datasets['train'])}")
print(f"Test samples: {len(image_datasets['test'])}")

Mounted at /content/drive
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 61.7 MB/s eta 0:00:00
  Created wheel for efficientnet-pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16424 sha256=995c624c741f586d646767f

In [ ]:
!pip install efficientnet_pytorch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 98.3 MB/s eta 0:00:00
  Created wheel for efficientnet_pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16426 sha256=1bc6f6cd90a30cd0ee65fda61875b2d902f8843cf1291889